# 00 — First Look at the H&M Dataset

Goal of this notebook: load the three core CSVs, check shapes, dtypes and date ranges, and produce a one-page mental model of the dataset.

**Do not model anything here.** This notebook is purely descriptive.

In [ ]:
import sys
from pathlib import Path

# Make `src` importable from the notebook
sys.path.append(str(Path.cwd().parent))

import pandas as pd
import matplotlib.pyplot as plt

from src.data.loaders import load_articles, load_customers, load_transactions
from src import config

pd.set_option('display.max_columns', 50)

## 1. Articles (product catalogue)

In [ ]:
articles = load_articles()
print('Shape:', articles.shape)
articles.head()

In [ ]:
# Top product groups
articles['product_group_name'].value_counts().head(10)

## 2. Customers

In [ ]:
customers = load_customers()
print('Shape:', customers.shape)
customers.head()

In [ ]:
# Age distribution
customers['age'].plot(kind='hist', bins=50, figsize=(10, 4), title='Customer age distribution')
plt.xlabel('age')
plt.show()

## 3. Transactions (sample first to keep it light)

In [ ]:
# Load only 1M rows to start — full file has ~31M
transactions = load_transactions(nrows=1_000_000)
print('Shape:', transactions.shape)
print('Date range:', transactions['t_dat'].min(), '→', transactions['t_dat'].max())
transactions.head()

In [ ]:
# Daily transaction volume (sample)
transactions.groupby(transactions['t_dat'].dt.date).size().plot(
    figsize=(12, 4), title='Daily transactions (1M sample)'
)
plt.ylabel('# transactions')
plt.show()

## 4. Quick sanity checks

Things to confirm before moving on:
- [ ] Articles file has ~105K rows
- [ ] Customers file has ~1.3M rows
- [ ] Transactions full file covers Sept 2018 → Sept 2020
- [ ] All article_ids in transactions exist in articles
- [ ] All customer_ids in transactions exist in customers

In [ ]:
# Referential integrity check on the sample
missing_articles = set(transactions['article_id']) - set(articles['article_id'])
missing_customers = set(transactions['customer_id']) - set(customers['customer_id'])
print(f'Articles not in catalogue: {len(missing_articles)}')
print(f'Customers not in master:   {len(missing_customers)}')

## Next steps

Once this notebook runs cleanly:

1. Open `01_eda.ipynb` for deeper exploration (seasonality, top categories, price distribution).
2. Build the article × week aggregation needed by the demand model.
3. Set up MLflow tracking before starting `02_demand_prediction.ipynb`.